# Diagnosis BGE-M3 frozen · empat cara membaca fitur

Tiga teks tetap: **soal + jawaban referensi + respons siswa**. Target hanya **rater 0–100**.
BGE tidak dilatih. Vektor dihitung sekali, kemudian dipakai bersama oleh empat kondisi.
Ini diagnosis cabang BGE/readout, bukan pengganti eksperimen S1/S2/S3 atau bukti perbaikan S3.

1. **MLP asli** sebagai kontrol.
2. **MLP target-center**: kurangi rerata target **train fold**, tambahkan kembali pada prediksi.
3. **MLP StandardScaler**: skala tiap komponen fitur memakai statistik **train fold**, tanpa target-centering.
4. **Ridge**: alpha dipilih melalui inner CV, tidak memakai outer validation/test.

Struktur mengikuti notebook sebelumnya: setup → upload → audit data → split → fitur → CV → evaluasi → lima Excel + ZIP.
Pilih **T4 GPU** untuk mempercepat ekstraksi BGE. Head dijalankan **CPU float32** secara sengaja untuk perbandingan yang konsisten; CPU juga dapat mengekstrak BGE, tetapi lebih lambat.
Tanpa clone dan tanpa Google Drive. Jalankan dari atas dengan **Run all**, lalu pilih `fix_dataset_300.xlsx` saat diminta.

## 1 · Ruang kerja dan setup

Semua pustaka dan kode ada di notebook. Worker numerik terpisah menghindari benturan NumPy kernel Colab. Torch bawaan runtime dipertahankan. Data, cache dan checkpoint hanya pada disk lokal; unduh hasil sebelum runtime berakhir.

In [ ]:
import os, sys, json, subprocess, html, secrets, atexit, argparse, hashlib, venv
from pathlib import Path
from IPython.display import display, HTML
try:
    from packaging.requirements import Requirement
except ImportError:
    from pip._vendor.packaging.requirements import Requirement
IN_COLAB = 'google.colab' in sys.modules or Path('/content').is_dir()
default_workspace='/content/AES_300_BGE_Frozen_Readout_Colab' if IN_COLAB else str(Path.cwd()/'outputs/AES_300_BGE_Frozen_Readout_Colab')
ROOT=Path(os.environ.get('AES300_WORKSPACE',default_workspace)).expanduser().absolute()
ROOT=ROOT.resolve()
if ROOT==Path('/content/drive') or Path('/content/drive') in ROOT.parents:
    raise ValueError('Workspace harus lokal; diagnostik BGE tidak menulis Google Drive.')
ROOT.mkdir(parents=True,exist_ok=True)
def prepare_local_storage(workspace, in_colab):
    storage=Path(workspace).expanduser().resolve()
    if storage==Path('/content/drive') or Path('/content/drive') in storage.parents:
        raise ValueError('diagnostik BGE hanya menyimpan lokal; pilih workspace di luar Google Drive.')
    storage.mkdir(parents=True,exist_ok=True)
    probe=storage/'.aes_write_check';probe.write_text('local write check');probe.unlink()
    return storage

# Identitas kode notebook untuk membedakan hasil dari implementasi yang berbeda.
NOTEBOOK_CODE_SHA256='bba02fa45d79dfc965e95eea19239f604a1444275d797c4bde8b6543838f646a'
print('Ruang kerja AES siap. Seluruh kode tersedia di notebook ini.')
print('Versi Python:',sys.version.split()[0])

# Hanya pustaka runtime yang diperlukan; alat pengujian tidak dipasang untuk menjalankan AES.
REQUIREMENTS_TEXT=r'''
# Torch dipertahankan dari runtime Colab supaya CUDA tidak diganti wheel CPU.
# Tanpa torch, setup memasang torch 2.6.0 CPU sebelum file ini.
numpy==2.2.6
pandas==2.2.3
scipy==1.15.3
scikit-learn==1.6.1
# tokenizers0.19 (transformers4.44) tidak menyediakan wheel Python3.13.
transformers==4.44.2; python_version == '3.12'
transformers==4.48.3; python_version == '3.13'
openpyxl==3.1.5
'''
(ROOT/'requirements.dataset300.txt').write_text(REQUIREMENTS_TEXT)

def resolve_dataset_upload(workspace, configured='', search_paths=None, browser_upload=False):
    """Web Colab: pakai file yang ada, atau tampilkan Choose Files pada cell ini."""
    from pathlib import Path
    import hashlib
    import os
    import shutil
    workspace=Path(workspace).expanduser().resolve()
    destination=workspace/'data/dataset300/fix_dataset_300.xlsx'
    if configured:
        candidates=[Path(configured).expanduser().resolve()]
        if not candidates[0].is_file():
            raise FileNotFoundError('DATASET_FILE/AES300_DATASET belum ditemukan: '+str(candidates[0]))
    else:
        places=search_paths if search_paths is not None else [Path('/content'),Path.cwd(),workspace]
        candidates=[]
        for place in places:
            path=(Path(place)/'fix_dataset_300.xlsx').resolve()
            if path.is_file() and path not in candidates and path!=destination:
                candidates.append(path)
        if not candidates and destination.is_file():
            candidates=[destination]
    if not candidates and browser_upload:
        from google.colab import files
        destination.parent.mkdir(parents=True,exist_ok=True)
        print('Klik Choose Files, lalu pilih fix_dataset_300.xlsx dari komputer.',flush=True)
        previous=Path.cwd()
        try:
            os.chdir(destination.parent)
            uploaded=files.upload()
        finally:
            os.chdir(previous)
        if 'fix_dataset_300.xlsx' not in uploaded:
            raise ValueError('Pilih fix_dataset_300.xlsx. Jika dibatalkan, jalankan ulang cell setup ini.')
        candidates=[destination]
        del uploaded
    if not candidates:
        raise FileNotFoundError(
            'Dataset belum ada. Di web Colab, jalankan cell setup ini dan pilih '
            'fix_dataset_300.xlsx melalui Choose Files. Alternatif: panel Files/Berkas kiri '
            '→ Upload, lalu jalankan ulang cell ini. Jika lokasinya berbeda, isi DATASET_FILE.')
    hashes={hashlib.sha256(p.read_bytes()).hexdigest() for p in candidates}
    if len(hashes)>1:
        raise ValueError('Ada beberapa dataset berbeda: '+', '.join(map(str,candidates))+
                         '. Isi DATASET_FILE dengan path file yang ingin dipakai.')
    source=candidates[0]
    if not source.stat().st_size:
        raise ValueError('File upload masih kosong; tunggu upload selesai lalu jalankan ulang cell ini.')
    destination.parent.mkdir(parents=True,exist_ok=True)
    if source!=destination:
        shutil.copy2(source,destination)
    print('Dataset dari upload lokal:',source,'→',destination,flush=True)
    return destination


In [ ]:
# Pemeriksaan dan pemasangan pustaka tersedia langsung di sini.
INSPECT = r'''
import json, sys
from importlib import metadata, util
try:
    from packaging.requirements import Requirement
except ImportError:
    from pip._vendor.packaging.requirements import Requirement
roots=json.loads(sys.argv[1])+['torch>=2.6.0']
versions={}
for raw in roots:
    name=Requirement(raw).name
    try:versions[name]=metadata.version(name)
    except metadata.PackageNotFoundError:versions[name]=None
queue=[Requirement(raw) for raw in roots];seen=set();issues=[];install_requirements=[]
while queue:
    req=queue.pop()
    if req.marker and not req.marker.evaluate({'extra':''}):continue
    try:dist=metadata.distribution(req.name)
    except metadata.PackageNotFoundError:
        issues.append(str(req)+' belum tersedia');install_requirements.append(str(req));continue
    if not req.specifier.contains(dist.version,prereleases=True):
        issues.append(str(req)+' diperlukan; tersedia '+dist.version);install_requirements.append(str(req))
    name=dist.metadata['Name'].lower().replace('_','-')
    if name in seen:continue
    seen.add(name)
    queue.extend(Requirement(raw) for raw in (dist.requires or []))
print(json.dumps({'python':sys.version.split()[0],'runtime_version':list(sys.version_info[:2]),
 'versions':versions,'torch_present':util.find_spec('torch') is not None,
 'dependency_issues':sorted(set(issues)),'install_requirements':sorted(set(install_requirements)),'dependency_count':len(seen)}))
'''

IMPORTS = r'''
def configure_text_only_transformers():
    """Hindari import vision ops yang tidak diperlukan oleh RoBERTa/BGE/LoRA."""
    import os
    os.environ['USE_TORCH'] = '1'
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    from transformers.utils import import_utils
    # Transformers 4.44/4.48 mendeteksi torchvision dari keberadaan paket,
    # meskipun binary ops-nya tidak cocok dengan torch runtime. Notebook ini
    # hanya teks: jangan muat backend gambar melalui import tidak langsung pustaka AES.
    # Hanya worker AES yang diubah; paket Colab dan operator torch tetap utuh.
    if not hasattr(import_utils, '_torchvision_available'):
        raise RuntimeError('Versi Transformers berbeda dari versi notebook yang didukung.')
    import_utils._torchvision_available = False
    return {'transformers_backend': 'pytorch_text_only', 'torchvision_enabled': False}

configure_text_only_transformers()
import json
import torch, numpy, pandas, scipy, sklearn, openpyxl
from transformers import AutoModel, AutoTokenizer
available=torch.cuda.is_available()
print(json.dumps({'torch':torch.__version__,'cuda_available':available,
 'device':torch.cuda.get_device_name(0) if available else 'CPU',
 'checkpoint_loaded':False,'imports_verified':True,'transformers_backend':'pytorch_text_only'}))
'''

def environment_folder(root, version):
    version=tuple(version[:2])
    if version not in ((3,12),(3,13)):
        raise RuntimeError('Setup AES mendukung Python 3.12 atau 3.13. Versi lain belum diuji.')
    return Path(root)/'outputs'/('dataset300_env_text_e6_gpu_reuse' if version==(3,12) else 'dataset300_env_py313_text_e6_gpu_reuse')


# Pilih versi pustaka sesuai Python aktif agar aturan Python 3.12 dan 3.13 tidak tertukar.
def active_requirements(path, version):
    """Hanya aturan untuk versi Python aktif; tidak mencoba semua versi Transformers."""
    selected=[]
    for line in Path(path).read_text().splitlines():
        line=line.split('#',1)[0].strip()
        if not line:continue
        req=Requirement(line)
        if req.marker and not req.marker.evaluate({'python_version':'.'.join(map(str,version[:2])), 'extra':''}):continue
        selected.append(req.name+str(req.specifier))
    return selected


class SetupError(RuntimeError):
    pass


# Memeriksa pustaka yang tersedia, menyiapkan versi yang diperlukan, lalu memvalidasi import sebelum melanjutkan.
def prepare_environment(argv=None):
    parser=argparse.ArgumentParser()
    parser.add_argument('--verify-python',help='Periksa worker yang disediakan pengguna tanpa memasang paket.')
    args=parser.parse_args(argv)
    folder=environment_folder(ROOT,sys.version_info)
    outputs=ROOT/'outputs';outputs.mkdir(parents=True,exist_ok=True)
    ready=outputs/'dataset300_setup.json';ready.unlink(missing_ok=True)
    logfile=outputs/'dataset300_setup.log'
    requirements=active_requirements(ROOT/'requirements.dataset300.txt',sys.version_info)
    env={**os.environ,'HF_HOME':str(outputs/'cache/huggingface'),'PIP_DISABLE_PIP_VERSION_CHECK':'1'}
    with logfile.open('w') as log:
        def execute(command,stage):
            log.write(stage+'\n'+json.dumps(command)+'\n');log.flush()
            process=subprocess.run(command,cwd=ROOT,env=env,stdout=subprocess.PIPE,stderr=subprocess.PIPE,text=True)
            log.write(process.stdout or '');log.write(process.stderr or '');log.flush()
            if process.returncode:
                details='\n'.join((process.stderr or process.stdout or 'Proses gagal tanpa rincian.').splitlines()[-18:])
                raise SetupError(stage+' gagal.\n'+details+'\nRincian lengkap: '+str(logfile))
            return process.stdout
        def inspect(python):
            result=json.loads(execute([python,'-c',INSPECT,json.dumps(requirements)],'Pemeriksaan versi dan dependensi'))
            if result['runtime_version']!=list(sys.version_info[:2]):
                raise SetupError('Versi Python worker berbeda dari runtime. Gunakan lingkungan untuk versi runtime ini.')
            return result

        print('1/3 · Memeriksa lingkungan dan versi pustaka AES...',flush=True)
        if args.verify_python:
            # abspath menjaga symlink bin/python venv; resolve akan memakai Python global.
            python=str(Path(args.verify_python).absolute())
        else:
            if folder.exists() and not (folder/'pyvenv.cfg').is_file():
                raise SetupError('Folder target bukan lingkungan Python AES; tidak ditimpa.')
            if not (folder/'bin/python').exists():
                # Torch CUDA tetap berasal dari runtime. Paket AES yang berbeda dipasang lokal.
                venv.EnvBuilder(with_pip=False,system_site_packages=True).create(folder)
            python=str(folder/'bin/python')
        before=inspect(python)
        torch_version=before['versions'].get('torch')
        if torch_version and not Requirement('torch>=2.6.0').specifier.contains(torch_version,prereleases=True):
            raise SetupError('PyTorch minimal 2.6.0 diperlukan. Pilih runtime Colab yang lebih baru; torch bawaan tidak diganti.')
        if not before['torch_present']:
            raise SetupError('PyTorch bawaan runtime belum tersedia. Pilih runtime Colab yang baru; setup ini tidak memasang wheel CPU.')
        installed=False
        after=before
        if after['dependency_issues']:
            if args.verify_python:
                raise SetupError('Worker yang dipilih belum memenuhi kebutuhan AES. Rincian: '+str(logfile))
            print('2/3 · Menyiapkan pustaka AES tanpa mengganti PyTorch runtime...',flush=True)
            # --no-deps mencegah pustaka AES menarik ulang torch. Dependensi lain
            # diperiksa ulang dari metadata dan dipasang hanya jika diperlukan.
            pip=[sys.executable,'-m','pip','--python',python,'install','--ignore-installed','--no-deps','--no-input']
            for attempt in range(8):
                if not after['dependency_issues']:
                    break
                pending=after['install_requirements']
                if not pending or any(Requirement(raw).name.lower().replace('_','-')=='torch' for raw in pending):
                    raise SetupError('Dependensi meminta perubahan PyTorch. Gunakan runtime Colab yang kompatibel; torch bawaan tidak ditimpa.')
                execute(pip+pending,'Pemasangan pustaka AES privat, putaran '+str(attempt+1))
                installed=True
                after=inspect(python)
                if after['versions'].get('torch')!=torch_version:
                    raise SetupError('Versi PyTorch berubah selama setup; training tidak dimulai.')
        else:
            print('2/3 · Versi pustaka sudah cocok; tidak perlu instalasi ulang.',flush=True)
        if after['dependency_issues']:
            raise SetupError('Dependensi AES belum cocok: '+'; '.join(after['dependency_issues'][:3])+'. Rincian: '+str(logfile))
        # Jangan mengganti torch yang terpasang tetapi rusak dengan wheel CPU secara diam-diam.
        print('3/3 · Memeriksa import dan perangkat perhitungan...',flush=True)
        checked=json.loads(execute([python,'-c',IMPORTS],'Pemeriksaan import AES dan PyTorch'))
        result={**after,**checked,'python_version':after['python'],'python':python,
                'packages_action':'installed' if installed else 'reused','torch_reused':before['torch_present'],
                'log_file':str(logfile),'requirements_sha256':hashlib.sha256((ROOT/'requirements.dataset300.txt').read_bytes()).hexdigest()}
        ready.write_text(json.dumps(result,indent=2)+'\n')
    print('Persiapan AES selesai. Versi, dependensi, dan import sudah diperiksa.',flush=True)
    return result


In [ ]:
# Penghubung proses Python: menjaga variabel antar-cell %%aes tanpa file pendukung proyek.
WORKER_SOURCE=r'''
import json, sys, traceback, types
from pathlib import Path

prefix, workspace = sys.argv[1:3]
module = types.ModuleType('__aes_notebook__')
sys.modules[module.__name__] = module
namespace = module.__dict__
namespace['ROOT'] = Path(workspace)

def event(kind, **values):
    print(prefix+json.dumps({'kind':kind, **values},ensure_ascii=False,allow_nan=False),flush=True)

def show(frame, title='', rows=8, columns=8):
    # Fungsi tampilan diisi oleh cell notebook; tidak diimpor dari file proyek.
    import pandas as pd
    frame = frame if isinstance(frame,pd.DataFrame) else pd.DataFrame(frame)
    view = namespace['preview_table'](frame.columns,frame.head(rows).to_dict('records'),title,
                                    row_count=len(frame),max_rows=rows,max_columns=columns)
    view['rows'] = json.loads(pd.DataFrame(view['rows']).to_json(orient='values',force_ascii=False,default_handler=str))
    event('table',**view,html=namespace['render_table'](view))

namespace['show'] = show
event('ready',python=sys.version.split()[0])
for line in sys.stdin:
    try:
        payload = json.loads(line)
        if payload.get('command') == 'close':break
        exec(compile(payload['code'],payload.get('filename','<aes-cell>'),'exec'),namespace)
    except BaseException:
        event('error',traceback=traceback.format_exc())
    else:
        event('done')
'''


In [ ]:
DATASET_FILE=''  # Biasanya otomatis dari /content; isi path lengkap jika berbeda.
PYTHON = os.environ.get('AES300_PYTHON','')
def show_installation_log():
    path=ROOT/'outputs/dataset300_setup.log'
    contents=path.read_text() if path.is_file() else 'Log belum tersedia.'
    display(HTML('<details style="background:#f8fafc;color:#172035;padding:12px;border:1px solid #94a3b8">'
                 '<summary>Rincian instalasi — buka jika diperlukan</summary>'
                 '<pre style="max-height:320px;overflow:auto;white-space:pre-wrap;color:#172035">'
                 +html.escape(contents)+'</pre></details>'))
try:
    ready=prepare_environment(['--verify-python',PYTHON] if PYTHON else [])
except Exception:
    show_installation_log()
    raise
if ready['runtime_version'] != list(sys.version_info[:2]) or not ready['imports_verified']:
    raise RuntimeError('Metadata setup tidak cocok runtime aktif.')
PYTHON=ready['python']
setup_rows=[
    ('Status','Siap — versi, dependensi, dan import AES lulus pemeriksaan'),
    ('Python',ready['python_version']),
    ('PyTorch',ready['torch']),
    ('Perangkat','BGE: GPU jika tersedia; semua head: CPU float32'),
    ('Pustaka AES','Versi yang cocok dipakai ulang' if ready['packages_action']=='reused'
                    else 'Versi khusus AES dipasang; pustaka bawaan tetap disimpan'),
    ('Pustaka utama','NumPy '+ready['versions']['numpy']+'; Transformers '+ready['versions']['transformers']),
    ('Model','Belum dimuat; tahap ini hanya menyiapkan lingkungan')]
status_html=''.join('<tr><th style="padding:9px;text-align:left;border:1px solid #94a3b8">'+html.escape(label)
                    +'</th><td style="padding:9px;border:1px solid #94a3b8">'+html.escape(value)+'</td></tr>'
                    for label,value in setup_rows)
display(HTML('<table style="background:#f8fafc!important;color:#172035!important;border-collapse:collapse">'
             +status_html+'</table>'))
show_installation_log()
PERSISTENT_ROOT=prepare_local_storage(ROOT,IN_COLAB)
print('Penyimpanan data, checkpoint dan Excel:',PERSISTENT_ROOT)
print('Checkpoint hanya lokal. Bisa dilanjutkan selama folder runtime masih ada; runtime dihapus berarti file lokal hilang. Unduh hasil sebelum menutup sesi.')
# Web Colab biasa: Choose Files muncul jika dataset belum diunggah.
DATA_PATH=resolve_dataset_upload(PERSISTENT_ROOT,DATASET_FILE or os.environ.get('AES300_DATASET',''),browser_upload=IN_COLAB)
def display_aes_table(event):
    display(HTML(event['html']))
if '_aes_process' in globals() and _aes_process.poll() is None:
    _aes_process.terminate()
_aes_prefix='__AES_'+secrets.token_hex(12)+'__'
child_env={**os.environ,'HF_HOME':str(ROOT/'outputs/cache/huggingface'),
           'AES300_DATASET':str(DATA_PATH),'AES300_RESULTS':str(PERSISTENT_ROOT/'outputs/dataset300_inline'),
           'AES300_REQUIRE_GPU':'0',
           'TOKENIZERS_PARALLELISM':'false',
           'OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2','PYTHONUNBUFFERED':'1'}
child_env.pop('PYTHONPATH',None)
_aes_process=subprocess.Popen([PYTHON,'-u','-c',WORKER_SOURCE,_aes_prefix,str(ROOT)],
                              cwd=ROOT,env=child_env,stdin=subprocess.PIPE,stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT,text=True,bufsize=1)
while True:
    line=_aes_process.stdout.readline()
    if not line:
        raise RuntimeError('Worker gagal mulai; lihat log setup.')
    if line.startswith(_aes_prefix):
        print('Proses perhitungan siap. Python',json.loads(line[len(_aes_prefix):])['python']);break
    print(line,end='')
def aes_cell(line,cell):
    if _aes_process.poll() is not None:
        raise RuntimeError('Worker berhenti; jalankan setup dan cell berikutnya dari awal.')
    _aes_process.stdin.write(json.dumps({'code':cell,'filename':'<notebook-cell>'})+'\n')
    _aes_process.stdin.flush()
    while True:
        output=_aes_process.stdout.readline()
        if not output:
            raise RuntimeError('Worker terhenti; jika RAM penuh pilih runtime lebih besar.')
        if not output.startswith(_aes_prefix):
            print(output,end='',flush=True);continue
        event=json.loads(output[len(_aes_prefix):])
        if event['kind']=='done':break
        if event['kind']=='error':raise RuntimeError(event['traceback'])
        if event['kind']=='table':display_aes_table(event)
get_ipython().register_magic_function(aes_cell,'cell','aes')
atexit.register(lambda:_aes_process.terminate() if _aes_process.poll() is None else None)
print('Dataset siap. Jalankan cell %%aes berurutan.')


## 2 · Konfigurasi yang dikunci sebelum perbandingan

LR, dropout, jumlah update dan aturan early stopping MLP sama untuk tiga kondisi. Head selalu baru dengan inisialisasi yang sama pada seed/fold yang sama. Tiga seed adalah pengulangan, bukan ensemble. Ridge deterministik dihitung sekali; hasilnya tidak disalin seolah tiga pengulangan independen.

`PRIMARY_TOLERANCE=2` hanya memilih sumbu presisi untuk tabel Pareto. Tetap dilaporkan ±1, ±2, ±5 dan ±10; tidak ada klaim satu model menang semua ukuran. **Test disisihkan dan tidak dievaluasi dalam notebook diagnosis ini.**

In [ ]:
%%aes
import os, sys, json, math, hashlib, random, warnings, re, itertools, unicodedata, zipfile, subprocess, gc, logging, html
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from importlib.metadata import version
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn.functional import normalize
from torch.utils.data import DataLoader
def configure_text_only_transformers():
    """Hindari import vision ops yang tidak diperlukan oleh RoBERTa/BGE/LoRA."""
    import os
    os.environ['USE_TORCH'] = '1'
    os.environ['USE_TF'] = '0'
    os.environ['USE_FLAX'] = '0'
    from transformers.utils import import_utils
    # Transformers 4.44/4.48 mendeteksi torchvision dari keberadaan paket,
    # meskipun binary ops-nya tidak cocok dengan torch runtime. Notebook ini
    # hanya teks: jangan muat backend gambar melalui import tidak langsung PEFT.
    # Hanya worker AES yang diubah; paket Colab dan operator torch tetap utuh.
    if not hasattr(import_utils, '_torchvision_available'):
        raise RuntimeError('Versi Transformers berbeda dari versi notebook yang didukung.')
    import_utils._torchvision_available = False
    return {'transformers_backend': 'pytorch_text_only', 'torchvision_enabled': False}

configure_text_only_transformers()
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from scipy.stats import pearsonr
from sklearn.metrics import cohen_kappa_score
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from openpyxl import Workbook, load_workbook
from openpyxl.cell import WriteOnlyCell
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter

NOTEBOOK_CODE_SHA256='bba02fa45d79dfc965e95eea19239f604a1444275d797c4bde8b6543838f646a'
SCOPE='bge_frozen_readout_diagnostic'
REQUIRED=['Email address','response_number','response_text','soal','jawaban_referensi','rater']

In [ ]:
%%aes
# Ubah hanya di sini sebelum mulai; jangan memilih parameter berdasarkan test.
EPOCHS, FOLDS = 25, 5
SEEDS = [42, 123, 2026]
HEAD_LR = 1e-4
RIDGE_ALPHAS = [.01, .1, 1., 10., 100.]
PRIMARY_TOLERANCE = 2
BGE_DEVICE = 'auto'
EXPECTED_ROWS = 300
cfg = dict(dataset=os.environ['AES300_DATASET'], output=str(ROOT/'outputs/bge_readout'),
    sheet='Sheet1', expected_rows=EXPECTED_ROWS, score_min=0, score_max=100,
    test_fraction=.2, folds=FOLDS, split_seed=2026, split_candidates=128,
    epochs=EPOCHS, early_stopping_patience=5, effective_batch=8, micro_batch=1,
    warmup_ratio=.1, gradient_clip=1., dropout=.1, weight_decay=.01,
    bge_max_length=1024, bge_overlap=128, bge_batch=4, device=BGE_DEVICE,
    context_max_tokens=192, question_max_tokens=64, input_format='single_sequence',
    bge=dict(checkpoint='BAAI/bge-m3', revision='5617a9f61b028005a4858fdac845db406aefb181'))
diagnostic = dict(seeds=SEEDS, lr=HEAD_LR, alphas=RIDGE_ALPHAS, primary_tolerance=PRIMARY_TOLERANCE)
if EPOCHS < 1 or FOLDS < 3:
    raise ValueError('Perlu epoch positif dan sedikitnya3fold untuk inner CV Ridge.')
torch.set_num_threads(2)
print('Ekstraksi BGE:',BGE_DEVICE,'| Head: CPU float32 | Data test dikunci',flush=True)

## 3 · Baca dan audit dataset

Enam header seperti eksperimen sebelumnya. Email/duplikat hanya membentuk kelompok agar tidak menyeberang partisi, bukan input model. Nilai kosong atau tidak sah menghentikan proses; tidak diisi otomatis.

In [ ]:
%%aes
def clean(text, normalize_punctuation=False):
    text = unicodedata.normalize('NFC', text).replace('\r\n','\n').replace('\r','\n')
    text = ''.join(c for c in text if c in '\n\t' or unicodedata.category(c) != 'Cc')
    if normalize_punctuation:
        text = text.translate(str.maketrans({'“':'"','”':'"','‘':"'",'’':"'",'–':'-','—':'-'})).replace('…','...')
    return text


def digest(path):
    # Checkpoint besar diperiksa bertahap supaya tidak dimuat seluruhnya ke RAM.
    checksum = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            checksum.update(chunk)
    return checksum.hexdigest()


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name+'.pending')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False)+'\n', encoding='utf-8')
    temporary.replace(path)


@dataclass
class Bundle:
    rows: list
    y: np.ndarray
    groups: np.ndarray
    source_sha256: str
    summary: dict

    @property
    def texts(self):
        return [r['text'] for r in self.rows]


def read_dataset(cfg):
    path = Path(cfg['dataset'])
    source_hash = digest(path)
    book = load_workbook(path, read_only=True, data_only=True)
    try:
        if cfg['sheet'] not in book.sheetnames:
            raise ValueError(f"Tab {cfg['sheet']} tidak tersedia: {book.sheetnames}")
        data = iter(book[cfg['sheet']].iter_rows(values_only=True))
        header = [str(x).strip() if x is not None else None for x in next(data)]
        if any(header.count(x) != 1 for x in REQUIRED):
            raise ValueError('Enam header wajib hilang atau duplikat: '+', '.join(REQUIRED))
        rows = []
        for excel_row, values in enumerate(data, 2):
            if all(v is None for v in values):
                continue  # Hanya baris format kosong, bukan label/jawaban yang hilang.
            raw = {k: values[i] for i, k in enumerate(header) if k is not None}
            if any(raw[k] is None or not str(raw[k]).strip() for k in REQUIRED):
                raise ValueError(f'Baris {excel_row}: kolom wajib kosong: '+', '.join(k for k in REQUIRED if raw[k] is None)+'; tidak diisi otomatis.')
            score = raw['rater']
            if isinstance(score, bool) or not isinstance(score, (int, float)) or not math.isfinite(score):
                raise ValueError(f'Baris {excel_row}: rater harus angka skor guru, bukan teks/ID.')
            if int(score) != score or not cfg['score_min'] <= score <= cfg['score_max']:
                raise ValueError(f'Baris {excel_row}: skor bukan kategori integer dalam rentang yang dinyatakan.')
            for key in ('soal','jawaban_referensi'):
                if not isinstance(raw[key],str) or not raw[key].strip():
                    raise ValueError(f'Baris {excel_row}: {key} harus teks yang tidak kosong.')
            if not isinstance(raw['response_text'], str):
                raise ValueError(f'Baris {excel_row}: jawaban bukan teks.')
            rows.append({'row_id': f'row_{excel_row-1:04d}', 'source_row': excel_row,
                         'text': clean(raw['response_text']), 'raw_text': raw['response_text'],
                         'email_group': unicodedata.normalize('NFC', str(raw['Email address'])).strip().casefold(),
                         'question': clean(raw['soal']), 'reference': clean(raw['jawaban_referensi']),
                         'response_number': str(raw['response_number']), 'score': int(score)})
    finally:
        book.close()
    if cfg.get('expected_rows') is not None and len(rows) != cfg['expected_rows']:
        raise ValueError(f"Diharapkan {cfg['expected_rows']} baris, ditemukan {len(rows)}; tidak dipotong otomatis.")
    if not rows:
        raise ValueError('Dataset kosong')
    parent = list(range(len(rows)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    def union(a, b):
        parent[find(b)] = find(a)

    email_first, duplicate_sets = {}, {}
    for i, row in enumerate(rows):
        email = row['email_group']
        if email in email_first:
            union(email_first[email], i)
        email_first[email] = i
        # Normalisasi ini HANYA audit kesamaan, bukan teks masukan encoder.
        normalized = ' '.join(row['text'].casefold().split())
        key = hashlib.sha256(normalized.encode()).hexdigest()
        duplicate_sets.setdefault(key, []).append(i)
    for indices in duplicate_sets.values():
        for i in indices[1:]:
            union(indices[0], i)
    roots = {root: i for i, root in enumerate(sorted({find(i) for i in range(len(rows))}))}
    groups = np.asarray([roots[find(i)] for i in range(len(rows))], dtype=int)
    y = np.asarray([r['score'] for r in rows], dtype=float)
    summary = {'scope': SCOPE, 'rows': len(rows), 'target_columns': ['rater'], 'model_outputs': 1,
               'score_range_observed': [float(y.min()), float(y.max())],
               'score_range_declared': [cfg['score_min'], cfg['score_max']],
               'scale_basis': '0–100 mengikuti notebook unggahan; deskriptor rubrik guru tidak tersedia.',
               'email_groups': len(email_first), 'connected_groups': len(roots),
               'questions': len({r['question'] for r in rows}),
               'duplicate_sets': sum(len(v) > 1 for v in duplicate_sets.values()),
               'cross_email_duplicate_sets': sum(len({rows[i]['email_group'] for i in v}) > 1 for v in duplicate_sets.values()),
               'source_sha256': source_hash, 'four_trait_labels_available': False,
               'four_rubric_report': 'Arsitektur empat output memungkinkan, tetapi empat skor tidak dapat ditentukan dari satu skor guru. Perlu anotasi manusia per dimensi; skor tidak disalin/dipecah otomatis.',
               'input_features': ['soal','response_text','jawaban_referensi'],
               'grouping_note': 'Email dipakai sebagai kelompok pencegah kebocoran; tidak membuktikan identitas terverifikasi. Duplikat lintas email disatukan. Semua skenario memakai soal/referensi dan jawaban. Email hanya kelompok; nilai guru hanya target, bukan fitur encoder.'}
    return Bundle(rows, y, groups, source_hash, summary)


def run_folder(cfg, bundle):
    code_hash = NOTEBOOK_CODE_SHA256
    semantic = {k: v for k, v in cfg.items() if k not in ('dataset', 'output', 'trace_position', 'vector_source', 'new_answer', 'focus_word', 'new_question', 'new_reference')}
    key = hashlib.sha256(json.dumps({'source': bundle.source_sha256, 'config': semantic, 'code': code_hash}, sort_keys=True).encode()).hexdigest()[:16]
    folder = Path(cfg['output']) / key
    folder.mkdir(parents=True, exist_ok=True)
    dump(folder / 'effective_config.json', cfg)
    return folder

In [ ]:
%%aes
# Bantuan membaca tabel: penamaan kolom, angka, dan penjelasan kata berulang.
LABELS = {'row_id':'ID Jawaban','word':'Kata','covered':'Tercakup','word_index':'Urutan Kata',
          'token_id':'ID Token','token':'Token','position':'Posisi Token','window':'Bagian Teks',
          'attention_mask':'Dibaca Model','skor_guru':'Nilai Guru','prediksi_mentah':'Prediksi Skor',
          'kategori_salinan':'Skor Dibulatkan','di_luar_rentang':'Di Luar Rentang',
          'Skenario':'Skenario','Fold':'Fold','AC2':'Gwet AC2','RMSE':'RMSE (poin)',
          'out_of_range_fraction':'Di Luar Rentang (%)','Status':'Keterangan',
          'teks_model':'Jawaban Siswa','teks_asli':'Teks Asli','score':'Nilai Guru',
          'role':'Penggunaan Data','connected_group':'Kelompok Jawaban',
          'input_format':'Format Input','response_text':'Jawaban Siswa','rater':'Nilai Guru','soal':'Soal',
          'sentence_index':'Urutan Kalimat','text':'Kalimat','fold':'Fold','scenario':'Skenario',
          'encoder':'Encoder','vector':'Panjang Vektor','head':'Lapisan Prediksi','output':'Jumlah Skor',
          'used_for':'Digunakan untuk','id':'Kode','title':'Judul',
          'rows':'Jumlah Jawaban','file':'File Hasil','stage':'Isi File','status':'Keadaan',
          'sheets_and_shapes':'Tabel yang Tersimpan','epoch':'Epoch',
          'train_mse_scaled':'Loss Latih (MSE)','target_loss':'Nilai untuk Menghitung Loss',
          'best_epoch_so_far':'Epoch Terbaik Sejauh Ini','epochs_without_improvement':'Epoch Tanpa Perbaikan',
          'selisih_mentah':'Selisih Prediksi − Guru','berubah':'Teks Dibersihkan?',
          'token_index':'Urutan Token','start':'Awal Teks','end':'Akhir Teks',
          'token_indices':'Urutan Potongan Token','token_ids':'ID Potongan Token',
          'covered_tokens':'Token yang Terbaca','total_tokens':'Jumlah Token',
          'window_coverage':'Jumlah Bagian yang Membaca Token',
          'occurrence_indices':'Posisi Kata yang Dibandingkan','same_token_ids':'ID Token Sama?',
          'initial_cosine':'Kemiripan Sebelum Konteks (Cosine)',
          'initial_max_abs_difference':'Selisih Terbesar Sebelum Konteks',
          'contextual_cosine':'Kemiripan Setelah Konteks (Cosine)',
          'contextual_max_abs_difference':'Selisih Terbesar Setelah Konteks',
          'contextual_allclose_1e-6':'Vektor Setelah Konteks Sama?',
          'undefined_reasons':'Alasan Metrik Tidak Tersedia','authors':'Penulis','year':'Tahun',
          'url':'Tautan','limitation':'Batas Penerapan','library':'Pustaka','version':'Versi',
          'name':'Pengaturan','value':'Nilai','field':'Keterangan','target':'Jenis Nilai',
          'Posisi Kata 1':'Posisi Kata A','Posisi Kata 2':'Posisi Kata B',
          'ID Token Sama':'ID Token Sama?','Vektor Awal Sama':'Vektor Sebelum Konteks Sama?',
          'Vektor Akhir Sama':'Vektor Setelah Konteks Sama?',
          'Kemiripan Vektor Akhir':'Kemiripan Setelah Konteks (Cosine)','Apa Artinya':'Penjelasan'}

REPEATED_COLUMNS = ['Model','Kata','Posisi Kata A','Posisi Kata B','ID Token Sama?',
                    'Vektor Sebelum Konteks Sama?','Vektor Setelah Konteks Sama?',
                    'Kemiripan Setelah Konteks (Cosine)','Penjelasan']

MODELS = {'S1':'RoBERTa','S2':'BGE-M3','S3':'Fusion RoBERTa + BGE-M3'}

ROLES = {'development':'Data latih (termasuk validasi silang)','test':'Data uji',
         'train':'Latih','validation':'Validasi'}

REPEATED_NOTES = [
    'Satu baris membandingkan kata yang sama pada dua posisi di jawaban contoh. Posisi dihitung mulai 1.',
    'Sebelum konteks = vektor dari tabel kosakata model. Setelah konteks = vektor setelah model membaca posisi dan kata di sekitarnya.',
    'Cosine mengukur kemiripan arah vektor: mendekati 1 berarti sangat mirip. Nilai yang terlihat 1 setelah pembulatan belum membuktikan seluruh angka vektornya sama.',
    'Kolom Vektor ... Sama? memeriksa selisih setiap komponen, dengan batas 0,000001. Ini batas perbandingan angka komputer, bukan tolerance skor AES.',
]

METRIC_NOTES = [
    'S1 = RoBERTa; S2 = BGE-M3; S3 = fusion keduanya. RMSE memakai satuan poin skor: lebih kecil berarti kesalahan lebih kecil.',
    'Pearson menunjukkan hubungan perubahan skor; QWK dan Gwet AC2 menunjukkan kesepakatan skor kategori. Korelasi tinggi saja belum menjamin prediksi mendekati nilai guru.',
    'Tanda — berarti hasil tidak tersedia, bukan nol. Baca keempat metrik bersama-sama.',
]

REPEATED_HEADERS=REPEATED_COLUMNS

LABELS.update({'input_segment':'Bagian Input','question_tokens':'Token Soal Awal',
    'question_kept':'Token Soal Dibaca','reference_tokens':'Token Referensi Awal',
    'reference_kept':'Token Referensi Dibaca','question_truncated':'Soal Terpotong?',
    'reference_truncated':'Referensi Terpotong?','context_tokens_kept':'Token Konteks Dibaca',
    'content_tokens':'Token Jawaban','capacity':'Kapasitas Jawaban per Window',
    'stride':'Langkah Window','starts':'Posisi Awal Window','windows':'Jumlah Window',
    'pair_special_tokens':'Token Khusus Pasangan','answer_tokens_uncovered':'Token Jawaban Tidak Tercakup',
    'context_max_tokens':'Batas Token Konteks Semua Encoder','question_max_tokens':'Batas Token Soal Semua Encoder'})

def friendly_header(name):
    match = re.fullmatch(r'v_(\d+)', str(name))
    return 'Komponen '+str(int(match[1])+1) if match else LABELS.get(name, str(name).replace('_',' '))


def friendly_value(name, value):
    if hasattr(value,'item'):
        try:value=value.item()
        except ValueError:pass
    if name == 'row_id' and isinstance(value,str):
        match = re.fullmatch(r'row_(\d+)',value)
        return f'J{int(match[1]):03d}' if match else value
    if name == 'role':
        return ROLES.get(value,value)
    if name == 'status':
        return {'ACTUAL EXECUTION':'Dihasilkan saat kode dijalankan',
                'DESIGN / CONFIGURATION':'Rancangan dan pengaturan model'}.get(value,value)
    if name == 'attention_mask' and value in (0,1):
        return 'Ya' if value else 'Tidak'
    if name == 'out_of_range_fraction' and isinstance(value,(int,float)):
        return value*100
    if name in ('word_index','sentence_index','token_index','position','window') and isinstance(value,(int,float)):
        return value+1
    if isinstance(value,bool):
        return 'Ya' if value else 'Tidak'
    return value


def repeated_word_rows(records,model):
    for row in records:
        positions=row.get('occurrence_indices')
        if isinstance(positions,str):positions=json.loads(positions)
        positions=positions or []
        same_ids=row.get('same_token_ids')
        initial=row.get('initial_max_abs_difference')
        same_initial=None if initial is None else abs(initial)<=1e-6
        same_context=row.get('contextual_allclose_1e-6')
        if same_ids is False:explanation='Kata sama, tetapi potongan token berbeda. Perhatikan tokenisasinya sebelum membandingkan vektor.'
        elif same_ids is None:explanation='Informasi kesamaan ID token belum tersedia.'
        elif same_context is None:explanation='Hasil perbandingan vektor akhir belum tersedia.'
        elif same_context:explanation='ID token sama; vektor akhir juga sama dalam toleransi numerik 0,000001.'
        else:explanation='ID token sama, tetapi vektor setelah membaca konteks berbeda. Posisi dan kata di sekitarnya dapat memengaruhi vektor.'
        label={'roberta':'RoBERTa','bge':'BGE-M3'}.get(model,model)
        yield [label,row.get('word'),positions[0]+1 if positions else None,positions[1]+1 if len(positions)>1 else None,
               friendly_value('',same_ids),friendly_value('',same_initial),friendly_value('',same_context),
               row.get('contextual_cosine'),explanation]

def preview_table(headers, records, title='', row_count=None, max_rows=8, max_columns=8):
    """Bentuk preview tanpa mengubah DataFrame/Excel sumber atau menghitung ulang model."""
    headers=list(headers)
    records=list(itertools.islice(records,max_rows))
    count=len(records) if row_count is None else row_count
    notes=[]
    original_columns=len(headers)
    if 'occurrence_indices' in headers:
        values=list(repeated_word_rows(records,''))
        # Nama model hanya ditampilkan bila memang diberikan oleh pemanggil.
        headers=REPEATED_HEADERS[1:]
        values=[r[1:] for r in values]
        notes=REPEATED_NOTES
        empty='Belum ada pasangan kata berulang yang bisa dibandingkan. Ini bukan error; lihat tabel vektor kata di atas.'
    elif {'file','stage','sheets_and_shapes'}.issubset(headers):
        # Dictionary ukuran tabel adalah rincian internal, bukan isi laporan belajar.
        headers=['File Hasil','Isi File','Keadaan']
        values=[[r.get('file'),r.get('stage'),friendly_value('status',r.get('status'))] for r in records]
        notes=['Daftar ini menunjukkan file yang sudah dibuat. Pilih lima laporan utama pada bagian unduhan untuk membaca hasilnya.']
        empty='Belum ada file hasil yang tersimpan.'
    else:
        selected=headers[:max_columns]
        values=[[friendly_value(key,row.get(key)) for key in selected] for row in records]
        headers=[friendly_header(key) for key in selected]
        empty='Belum ada baris hasil pada tahap ini. Tidak ada angka pengganti yang ditampilkan.'
        if 'Vektor Setelah Konteks Sama?' in headers or 'Vektor Akhir Sama' in headers:
            notes=REPEATED_NOTES
            empty='Belum ada pasangan kata berulang yang bisa dibandingkan. Ini bukan error; lihat tabel vektor kata di atas.'
        elif {'RMSE','Pearson','QWK','AC2'}.issubset(selected) or 'RMSE (poin)' in headers:
            notes=METRIC_NOTES
        elif 'prediksi_mentah' in selected or 'Prediksi S1' in headers:
            notes=['Prediksi Skor adalah nilai yang diberikan model. Nilai Guru adalah acuan jika tersedia.',
                   'Skor Dibulatkan adalah salinan untuk metrik kategori; prediksi asli tetap disimpan. Di Luar Rentang berarti prediksi melewati batas skor yang ditetapkan.']
        elif 'token_id' in selected or 'ID Token' in headers:
            notes=['Token adalah potongan teks yang dibaca model; satu kata bisa terpecah menjadi beberapa token. ID Token adalah nomor dalam kosakata model, bukan nilai siswa.',
                   'Posisi Token dan Bagian Teks dihitung mulai 1. Token khusus seperti <s> dan </s> menandai batas input.']
        components=sum(bool(re.fullmatch(r'v_\d+|Komponen \d+',str(h))) for h in selected)
        if components:
            notes=['Satu baris berisi satu vektor. Komponen 1, 2, dan seterusnya adalah angka penyusun vektor, bukan skor penilaian.',
                   'Tabel ini hanya memperlihatkan beberapa komponen pertama. Seluruh komponen tersimpan di Excel. Komponen yang sama nomornya pada RoBERTa dan BGE-M3 tidak memiliki arti yang otomatis sama.']
    return {'title':title,'columns':headers,'rows':values,'shape':[count,original_columns],
            'notes':notes,'empty_message':empty}


def cell_text(value):
    if value is None:return '—'
    if isinstance(value,bool):return 'Ya' if value else 'Tidak'
    if isinstance(value,float):
        return format(value,'.8g') if math.isfinite(value) else '—'
    if isinstance(value,dict):
        return '; '.join(f'{friendly_header(k)}: {cell_text(v)}' for k,v in value.items()) or '—'
    if isinstance(value,(list,tuple)):
        return ', '.join(cell_text(x) for x in value) or '—'
    return str(value)


def render_table(event):
    """HTML escapé dengan warna eksplisit agar terbaca pada Colab terang maupun gelap."""
    esc=lambda value:html.escape(cell_text(value))
    parts=['<section style="background:#f8fafc!important;color:#172035!important;padding:18px;border:1px solid #94a3b8;border-radius:12px;margin:12px 0">',
           '<h3 style="color:#172035!important;margin:0 0 10px">'+esc(event.get('title',''))+'</h3>']
    for note in event.get('notes',[]):parts.append('<p style="color:#334155!important;margin:6px 0">'+esc(note)+'</p>')
    if not event['rows']:
        parts.append('<p>'+esc(event.get('empty_message','Belum ada hasil.'))+'</p>')
    else:
        parts.append(f'<p style="color:#475569!important">Ditampilkan {len(event["rows"])} dari {event["shape"][0]} baris. Angka pada layar diringkas; nilai lengkap ada di Excel.</p>')
        th='background:#172035!important;color:#fff!important;padding:10px;border:1px solid #64748b;text-align:left'
        td='background:#f8fafc!important;color:#172035!important;padding:10px;border:1px solid #cbd5e1;vertical-align:top;min-width:90px;max-width:460px;white-space:normal;overflow-wrap:anywhere'
        parts.append('<div style="max-height:420px;overflow:auto"><table style="border-collapse:collapse"><thead><tr>')
        parts.extend('<th style="'+th+'">'+esc(x)+'</th>' for x in event['columns'])
        parts.append('</tr></thead><tbody>')
        for row in event['rows']:
            parts.append('<tr>'+''.join('<td style="'+td+'">'+esc(x)+'</td>' for x in row)+'</tr>')
        parts.append('</tbody></table></div>')
    parts.append('</section>')
    return ''.join(parts)

# Memeriksa nama GPU, status T4, dan VRAM; Colab berhenti sebelum training jika GPU belum aktif.
def check_gpu(require_gpu=False):
    """Periksa hardware yang benar-benar tersedia, tanpa memuat model pretrained."""
    available = torch.cuda.is_available()
    if not available:
        if require_gpu:
            raise RuntimeError('GPU belum aktif. Pilih Runtime → Change runtime type → T4 GPU, '
                               'lalu jalankan notebook dari atas. Training Colab tidak dimulai di CPU.')
        return {'name': 'CPU', 'cuda_available': False, 'is_t4': False, 'memory_gib': None}
    name = torch.cuda.get_device_name(0)
    return {'name': name, 'cuda_available': True, 'is_t4': 'T4' in name.upper(),
            'memory_gib': round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2)}


In [ ]:
%%aes
bundle = read_dataset(cfg)
output = run_folder(cfg, bundle)
show(pd.DataFrame([bundle.summary]), 'Audit dataset asli')
show(pd.DataFrame([dict(row_id=r['row_id'], soal=r['question'], referensi=r['reference'],
                       respons=r['text'], rater=r['score']) for r in bundle.rows]), 'Tiga input dan satu label')
print('Sumber dataset SHA256:',bundle.source_sha256,flush=True)

## 4 · Development, test terkunci dan lima fold

Sama seperti sumber acuan: pilih split kelompok berdasarkan proporsi jumlah baris, tanpa label. Lima fold hanya di development. Test tidak memiliki prediksi pada hasil diagnosis ini. Inner CV Ridge memakai blok fold lain, dengan kelompok tetap terpisah.

In [ ]:
%%aes
def make_split(bundle, cfg):
    indices = np.arange(len(bundle.rows))
    splitter = GroupShuffleSplit(n_splits=cfg['split_candidates'], test_size=cfg['test_fraction'], random_state=cfg['split_seed'])
    # Pilih kedekatan JUMLAH BARIS dengan target 80:20, bukan skor/hasil model.
    # GroupShuffleSplit membagi kelompok; ukuran kelompok tidak selalu sama.
    candidates = list(splitter.split(indices, groups=bundle.groups))
    development, test = min(candidates, key=lambda pair: abs(len(pair[1])/len(indices)-cfg['test_fraction']))
    if len(set(bundle.groups[development])) < cfg['folds']:
        raise ValueError('Kelompok training tidak cukup untuk CV yang diminta.')
    folds = []
    for number, (a, b) in enumerate(GroupKFold(n_splits=cfg['folds']).split(development, groups=bundle.groups[development]), 1):
        training, validation = development[a], development[b]
        assert not set(bundle.groups[training]) & set(bundle.groups[validation])
        folds.append({'fold': number, 'train': training.tolist(), 'validation': validation.tolist()})
    assert not set(bundle.groups[development]) & set(bundle.groups[test])
    manifest = {'scope': SCOPE, 'source_sha256': bundle.source_sha256,
                'development': development.tolist(), 'test': test.tolist(), 'folds': folds,
                'development_rows': len(development), 'test_rows': len(test),
                'development_groups': len(set(bundle.groups[development])),
                'test_groups': len(set(bundle.groups[test])),
                'target_test_row_fraction': cfg['test_fraction'],
                'candidate_group_fraction': cfg['test_fraction'],
                'split_candidates': cfg['split_candidates'],
                'selection_uses_labels': False,
                'actual_test_row_fraction': len(test)/len(indices),
                'policy': 'Holdout kelompok: pilih kandidat deterministik terdekat target proporsi baris berdasarkan ukuran kelompok saja, tanpa label/hasil model. CV hanya di development. Email/duplikat tidak menyeberang. Mengukur kelompok baru pada pertanyaan ini, bukan unseen-prompt.',
                'test_used_for_selection': False}
    manifest['split_sha256'] = hashlib.sha256(json.dumps(manifest, sort_keys=True).encode()).hexdigest()
    return manifest


def contexts_from_bundle(bundle):
    """Ambil soal/referensi tanpa membaca target skor guru."""
    contexts = []
    for row in bundle.rows:
        question, reference = row.get('question'), row.get('reference')
        if not isinstance(question, str) or not question.strip():
            raise ValueError('Context pair memerlukan soal yang tidak kosong.')
        if not isinstance(reference, str) or not reference.strip():
            raise ValueError('Context pair memerlukan jawaban referensi yang tidak kosong.')
        contexts.append((clean(question), clean(reference)))
    return contexts

In [ ]:
%%aes
manifest = make_split(bundle, cfg)
dump(output/'split_manifest.json', manifest)
show(pd.DataFrame([dict(development=len(manifest['development']), test_locked=len(manifest['test']),
                       folds=len(manifest['folds']), development_groups=manifest['development_groups'],
                       test_groups=manifest['test_groups'])]), 'Partisi aktual')
print('Identitas split:',manifest['split_sha256'],flush=True)

## 5 · Tokenisasi dan BGE-M3 frozen

Resep fitur mengikuti single-sequence E21: soal+referensi+respons, mean CLS antar-window, lalu L2. Nilai guru tidak masuk tokenizer. BGE memakai revision terkunci, `eval()`, `requires_grad=False` dan `no_grad()`. Tidak ada RoBERTa yang dimuat dan tidak ada optimizer untuk encoder.

Cache diverifikasi dengan hash data/resep dan checksum fitur. Menjalankan ulang dengan data/resep sama memakai vektor yang sama.

In [ ]:
%%aes
def mean_by_essay(cls, mapping, n):
    sums=cls.new_zeros((n,cls.shape[-1])).index_add(0,mapping,cls)
    counts=cls.new_zeros(n).index_add(0,mapping,cls.new_ones(len(mapping)))
    if (counts==0).any(): raise ValueError('Esai tanpa window')
    return sums / counts[:,None]


def context_token_ids(context, tokenizer, cfg):
    """Sisakan anggaran untuk soal dan referensi sekaligus."""
    if not isinstance(context, (tuple, list)) or len(context) != 2:
        raise ValueError('Konteks harus pasangan (soal, jawaban referensi).')
    question, reference = context
    if not all(isinstance(t, str) and t.strip() for t in context):
        raise ValueError('Isi soal dan jawaban referensi sebelum memprediksi.')
    limit, qlimit = cfg['context_max_tokens'], cfg['question_max_tokens']
    if not 1 <= qlimit < limit:
        raise ValueError('Anggaran token soal harus lebih kecil daripada konteks.')
    # Prefix menjadi bagian input; nilai guru tidak pernah dimasukkan.
    qids = tokenizer.encode('Soal: '+question, add_special_tokens=False)
    rids = tokenizer.encode('\nJawaban referensi: '+reference, add_special_tokens=False)
    question_kept = qids[:qlimit]
    reference_kept = rids[:limit-len(question_kept)]
    return question_kept+reference_kept, {
        'question_tokens': len(qids), 'question_kept': len(question_kept),
        'reference_tokens': len(rids), 'reference_kept': len(reference_kept),
        'question_truncated': len(question_kept) < len(qids),
        'reference_truncated': len(reference_kept) < len(rids),
        'context_tokens_kept': len(question_kept)+len(reference_kept),
    }


def pair_content_windows(text, context, tokenizer, maximum=512, overlap=128, cfg=None):
    """Informasi dan window sama; E22 menambah separator resmi pasangan."""
    if cfg is None or cfg.get('input_format') not in ('single_sequence', 'context_pair'):
        raise ValueError('Pilih format input E21 atau E22 sebelum tokenisasi.')
    context_ids, context_info = context_token_ids(context, tokenizer, cfg)
    ids = tokenizer.encode(text, add_special_tokens=False)
    answer_prefix = tokenizer.encode('\nJawaban siswa:\n', add_special_tokens=False)
    if not ids:
        raise ValueError('Jawaban tidak menghasilkan token.')
    paired = cfg['input_format'] == 'context_pair'
    specials = tokenizer.num_special_tokens_to_add(pair=paired)
    # Cadangkan overhead terbesar untuk KEDUA eksperimen. Starts/cakupan sama.
    reserved = max(tokenizer.num_special_tokens_to_add(pair=False),
                   tokenizer.num_special_tokens_to_add(pair=True))
    capacity = maximum-len(context_ids)-len(answer_prefix)-reserved
    if not 0 <= overlap < capacity:
        raise ValueError('Konteks dan penanda jawaban menyisakan kapasitas di bawah overlap.')
    starts = [0]
    while starts[-1]+capacity < len(ids):
        starts.append(starts[-1]+capacity-overlap)
    chunks, answer_positions, segments = [], [], []
    for start in starts:
        answer = ids[start:start+capacity]
        second = answer_prefix+answer
        if paired:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids, second)
            layout = tokenizer.build_inputs_with_special_tokens([-1]*len(context_ids),
                     [-3]*len(answer_prefix)+[-2]*len(answer))
        else:
            prepared = tokenizer.build_inputs_with_special_tokens(context_ids+second)
            layout = tokenizer.build_inputs_with_special_tokens(
                     [-1]*len(context_ids)+[-3]*len(answer_prefix)+[-2]*len(answer))
        positions = [i for i, value in enumerate(layout) if value == -2]
        if (len(prepared) != len(layout) or len(prepared) > maximum
                or len(prepared)-len(context_ids)-len(second) != specials
                or [prepared[i] for i in positions] != answer):
            raise ValueError('Template tokenizer tidak konsisten dengan format input.')
        chunks.append({'input_ids':prepared, 'attention_mask':[1]*len(prepared)})
        answer_positions.append(positions)
        segments.append(['Soal/referensi' if v == -1 else 'Jawaban siswa' if v == -2
                         else 'Penanda jawaban' if v == -3 else 'Token khusus' for v in layout])
    stats = {**context_info, 'input_format':cfg['input_format'], 'content_tokens':len(ids),
             'capacity':capacity, 'stride':capacity-overlap, 'starts':starts,
             'windows':len(chunks), 'pair_special_tokens':specials,
             'reserved_special_tokens':reserved, 'answer_prefix_tokens':len(answer_prefix),
             'answer_positions':answer_positions, 'segments':segments, 'answer_tokens_uncovered':0}
    return chunks, stats


def pair_batch_windows(texts, contexts, tokenizer, device='cpu', maximum=512, overlap=128, cfg=None):
    """Susun batch pasangan tanpa menukar konteks antarjawaban."""
    if len(texts) != len(contexts) or not texts:
        raise ValueError('Jumlah konteks harus sama dengan jumlah jawaban.')
    chunks, mapping, stats = [], [], []
    for index, (text, context) in enumerate(zip(texts, contexts)):
        values, detail = pair_content_windows(text, context, tokenizer, maximum, overlap, cfg)
        chunks.extend(values)
        mapping.extend([index]*len(values))
        stats.append(detail)
    batch = tokenizer.pad(chunks, padding=True, return_tensors='pt')
    return {key: value.to(device) for key, value in batch.items()}, torch.tensor(mapping, device=device), stats


def paired_pretrained_values(bundle, cfg, role):
    """Mean CLS antar-window; BGE dinormalisasi L2 setelah agregasi."""
    device = device_name(cfg)
    model, tok = encoder(cfg,role).to(device), tokenizer(cfg,role)
    model.eval()
    vectors, window_vectors, ids, coverage = [], [], [], []
    with torch.no_grad():
        for index, (text,context) in enumerate(zip(bundle.texts,contexts_from_bundle(bundle))):
            windows,mapping,details = pair_batch_windows([text],[context],tok,device,
                cfg[role+'_max_length'],cfg[role+'_overlap'],cfg)
            # Satu jawaban diproses bertahap agar jumlah window tidak meledakkan memori.
            cls_parts=[]
            for start in range(0,len(mapping),cfg['bge_batch']):
                part={k:v[start:start+cfg['bge_batch']] for k,v in windows.items()}
                cls_parts.append(model(**part).last_hidden_state[:,0,:].float())
            cls=torch.cat(cls_parts,dim=0)
            value=mean_by_essay(cls,mapping,1)
            if role=='bge':value=torch.nn.functional.normalize(value,dim=1)
            vectors.append(value.cpu().numpy())
            window_vectors.append(cls.cpu().numpy())
            row_id=bundle.rows[index].get('row_id',f'inference_{index}')
            ids.extend(f'{row_id}_window_{j}' for j in range(len(cls)))
            coverage.append({k:v for k,v in details[0].items() if k not in ('answer_positions','segments')})
            if (index+1)%25==0 or index+1==len(bundle.rows):
                print(f'{role}: tiga input {index+1}/{len(bundle.rows)} jawaban selesai.',flush=True)
    values=np.concatenate(vectors).astype(np.float32)
    if role=='bge' and any(p.requires_grad for p in model.parameters()):
        raise ValueError('Encoder BGE harus frozen.')
    del model,tok
    release()
    return values,np.concatenate(window_vectors),ids,coverage

In [ ]:
%%aes
def device_name(cfg):
    requested = cfg['device']
    if requested == 'cuda' and not torch.cuda.is_available():
        raise RuntimeError('CUDA diminta tetapi tidak tersedia; pilih GPU di Colab atau device=cpu.')
    return ('cuda' if torch.cuda.is_available() else 'cpu') if requested == 'auto' else requested


def release():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


class ExpectedPoolerNotice(logging.Filter):
    """Hanya saring pemberitahuan dua bobot pooler yang memang tidak dipakai."""
    def filter(self, record):
        import ast
        import re
        message = record.getMessage()
        if not message.startswith('Some weights of the model checkpoint at ') or 'were not used when initializing' not in message:
            return True
        match = re.search(r"\['[^\n]+?\]", message)
        if match is None:
            return True
        try:
            keys = ast.literal_eval(match.group())
        except (ValueError, SyntaxError):
            return True
        return set(keys) != {'pooler.dense.bias', 'pooler.dense.weight'}


def encoder(cfg, role):
    # Ambil nama checkpoint dan revision model yang dipilih: roberta atau bge.
    if role != 'bge':
        raise ValueError('Diagnosis ini hanya memuat BGE frozen.')
    spec = cfg[role]
    logger = logging.getLogger('transformers.modeling_utils')
    notice = ExpectedPoolerNotice()
    logger.addFilter(notice)
    try:
        # Muat arsitektur dan bobot pretrained. Ini baru inisialisasi encoder, belum latihan memakai nilai guru.
        result = AutoModel.from_pretrained(spec['checkpoint'], revision=spec['revision'], add_pooling_layer=False)
    finally:
        logger.removeFilter(notice)
    # Periksa panjang vektor: RoBERTa 768 angka, BGE-M3 1024 angka.
    expected = 768 if role == 'roberta' else 1024
    if result.config.hidden_size != expected:
        raise ValueError(f'{role}: dimensi checkpoint tidak cocok dengan {expected}.')
    # Parameter RoBERTa ikut diperbarui; parameter BGE tidak ikut diperbarui.
    # RoBERTa boleh belajar; bobot encoder BGE tidak diperbarui.
    result.requires_grad_(False)
    return result


def tokenizer(cfg, role, fast=False):
    # Muat tokenizer dari checkpoint yang sama agar ID token cocok dengan kosakata model.
    result = AutoTokenizer.from_pretrained(cfg[role]['checkpoint'], revision=cfg[role]['revision'],
                                           use_fast=fast if role == 'roberta' else True)
    if role == 'roberta' and not fast and result.num_special_tokens_to_add(False) != 2:
        raise ValueError('RoBERTa slow tokenizer harus menambahkan dua special token.')
    return result

In [ ]:
%%aes
def bge_features(bundle, cfg, output):
    """Cache berubah jika soal, referensi, jawaban, atau resep token berubah."""
    contexts=contexts_from_bundle(bundle)
    contract={'model':cfg['bge'], 'texts':bundle.texts, 'contexts':contexts,
              'maximum':cfg['bge_max_length'],'overlap':cfg['bge_overlap'],
              'context_max_tokens':cfg['context_max_tokens'],'question_max_tokens':cfg['question_max_tokens'],
              'policy':'three_inputs_mean_CLS_then_L2_v2', 'input_format':cfg['input_format']}
    key=hashlib.sha256(json.dumps(contract,sort_keys=True,ensure_ascii=False).encode()).hexdigest()[:20]
    cache=Path(cfg['output'])/'cache'/f'bge_three_inputs_{key}.npz'
    metadata=cache.with_suffix('.json')
    if cache.exists() and metadata.exists():
        proof=json.loads(metadata.read_text())
        if proof['file_sha256']!=hashlib.sha256(cache.read_bytes()).hexdigest():
            raise ValueError('Cache BGE berubah; tidak dipakai.')
        with np.load(cache,allow_pickle=False) as data:values=data['features']
        reused=True
    else:
        values,_,_,coverage=paired_pretrained_values(bundle,cfg,'bge')
        cache.parent.mkdir(parents=True,exist_ok=True)
        np.savez_compressed(cache,features=values)
        proof={'model':cfg['bge'],'shape':list(values.shape),'frozen':True,
               'pooling':'mean paired-window CLS then L2','max_length':cfg['bge_max_length'],
               'feature_source':'soal + jawaban_referensi + response_text; no teacher score', 'input_format':cfg['input_format'],
               'answer_tokens_uncovered':sum(s['answer_tokens_uncovered'] for s in coverage),
               'context_truncated_rows':sum(s['question_truncated'] or s['reference_truncated'] for s in coverage),
               'file_sha256':hashlib.sha256(cache.read_bytes()).hexdigest()}
        dump(metadata,proof)
        reused=False
    if values.shape!=(len(bundle.rows),1024) or not np.isfinite(values).all() or not np.allclose(np.linalg.norm(values,axis=1),1,atol=1e-4):
        raise ValueError('Fitur BGE shape/normalisasi tidak sah.')
    dump(Path(output)/'bge_encoding.json',{**proof,'reused':reused})
    return torch.from_numpy(values)
bge = bge_features(bundle, cfg, output)
X = bge.numpy().astype(np.float32)
np.save(output/'bge_frozen.npy', X)
show(pd.DataFrame([dict(shape=str(X.shape), norm_min=float(np.linalg.norm(X,axis=1).min()),
    norm_max=float(np.linalg.norm(X,axis=1).max()), encoder_trained=False)]), 'Fitur yang dipakai semua kondisi')
print('BGE selesai; model encoder sudah dilepas dari memori. Selanjutnya hanya head CPU.',flush=True)

## 6 · Metrik prediksi asli

RMSE/MAE memakai skor kontinu asli, tanpa clipping untuk mempercantik error. QWK/Gwet AC2 memakai salinan kategori0–100 dengan tie.5ke bawah. Pearson atau agreement yang tidak terdefinisi ditandai, bukan diganti0. Jumlah dalam toleransi memiliki arti berbeda dari RMSE.

In [ ]:
%%aes
def gwet_ac2(teacher, predicted, categories):
    a=np.asarray(teacher);b=np.asarray(predicted);c=np.asarray(categories,dtype=float)
    if len(a)!=len(b) or len(a)<2: raise ValueError('AC2 perlu minimal dua pasangan')
    if not np.isin(a,c).all() or not np.isin(b,c).all(): raise ValueError('Kategori AC2 tidak sah')
    q=len(c); weights=1-((c[:,None]-c[None,:])/(c.max()-c.min()))**2
    ai=np.searchsorted(c,a); bi=np.searchsorted(c,b)
    pa=weights[ai,bi].mean()
    pi=(np.bincount(ai,minlength=q)+np.bincount(bi,minlength=q))/(2*len(a))
    pe=weights.sum()*np.sum(pi*(1-pi))/(q*(q-1))
    return float((pa-pe)/(1-pe))

def metrics(teacher, prediction, cfg):
    y, p = np.asarray(teacher, dtype=float), np.asarray(prediction, dtype=float)
    if y.ndim != 1 or y.shape != p.shape or len(y) == 0:
        raise ValueError('Metrik satu skor memerlukan dua vektor [N] yang sama dan tidak kosong.')
    if not np.isfinite(y).all() or not np.isfinite(p).all():
        raise ValueError('Label/prediksi tidak finite.')
    cats = np.arange(cfg['score_min'], cfg['score_max']+1)
    if not np.isin(y, cats).all():
        raise ValueError('Skor guru harus kategori integer pada rentang yang dinyatakan.')
    # Buat SALINAN skor sah untuk agreement; prediksi mentah tetap utuh.
    clipped = np.clip(p, cfg['score_min'], cfg['score_max'])
    # Tie .5 memilih kategori lebih rendah, sama dengan kontrak konversi proyek.
    ordinal = cats[np.abs(clipped[:, None]-cats[None, :]).argmin(axis=1)]
    result = {'RMSE': float(np.sqrt(np.mean((y-p)**2))), 'Pearson': None,
              'QWK': None, 'AC2': None, 'undefined_reasons': {},
              'out_of_range_fraction': float(np.mean((p < cats[0]) | (p > cats[-1])))}
    if len(y) > 1 and np.ptp(y) > 0 and np.ptp(p) > 0:
        result['Pearson'] = float(pearsonr(y, p).statistic)
    else:
        result['undefined_reasons']['Pearson'] = 'Perlu minimal dua pasangan dan kedua vektor tidak konstan.'
    if len(y) > 1:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            # QWK memakai jarak kategori kuadratik, bukan vektor embedding.
            qwk = cohen_kappa_score(y, ordinal, labels=cats, weights='quadratic')
        if np.isfinite(qwk):
            result['QWK'] = float(qwk)
        else:
            result['undefined_reasons']['QWK'] = 'Expected disagreement nol; undefined, bukan nol.'
        # AC2 memakai kategori yang sama dengan QWK dan koreksi peluang berbeda.
        result['AC2'] = gwet_ac2(y, ordinal, cats)
    else:
        result['undefined_reasons'].update(QWK='Perlu minimal dua pasangan.', AC2='Perlu minimal dua pasangan.')
    return result

## 7 · Empat kondisi pembanding

StandardScaler memakai rerata dan standar deviasi tiap komponen pada train fold; ini berbeda dari L2 per jawaban yang sudah dipakai BGE. Target-center hanya mengubah offset target dan mengembalikannya saat membaca skor. Keduanya diuji terpisah; tidak ditambahkan fitur coverage atau diganti LR di tengah eksperimen.

In [ ]:
%%aes
"""Runtime embedded by build_bge_readout_notebook.py; no encoder training."""
import copy
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

CONDITIONS = ('MLP_ASLI', 'MLP_TARGET_CENTER', 'MLP_STANDARD_SCALER', 'RIDGE')
DESCRIPTIONS = {
    'MLP_ASLI': 'BGE L2 → Linear1024/256 → GELU → dropout → Linear256/1',
    'MLP_TARGET_CENTER': 'Head sama; target dikurangi rerata train fold, ditambah kembali saat prediksi',
    'MLP_STANDARD_SCALER': 'Head sama; StandardScaler per komponen dipasang hanya pada train fold',
    'RIDGE': 'BGE L2 yang sama; Ridge alpha dipilih melalui inner CV berkelompok',
}
REPORT_BOOKS = ('01_Data_dan_Split.xlsx', '02_Vektor_BGE_dan_Transformasi.xlsx',
                '03_Prediksi_OOF.xlsx', '04_Evaluasi_dan_Duel.xlsx', '05_Riwayat_Pelatihan.xlsx')


def readout_config(values):
    result = dict(values)
    if not result['seeds'] or len(set(result['seeds'])) != len(result['seeds']):
        raise ValueError('SEEDS harus unik dan tidak kosong.')
    if any(type(s) is not int or s < 0 for s in result['seeds']):
        raise ValueError('Seed harus integer nonnegatif.')
    if not result['alphas'] or any(not math.isfinite(a) or a <= 0 for a in result['alphas']):
        raise ValueError('Alpha Ridge harus positif dan finite.')
    if result['primary_tolerance'] not in (1, 2, 5):
        raise ValueError('Toleransi utama harus 1, 2, atau 5 poin.')
    if not math.isfinite(result['lr']) or result['lr'] <= 0:
        raise ValueError('LR head tidak sah.')
    return result


def prepare_transform(X, y, training, condition):
    """No validation/test statistics enter either transformation."""
    ids = np.asarray(training, dtype=int)
    scaler = StandardScaler().fit(X[ids]) if condition == 'MLP_STANDARD_SCALER' else None
    offset = float(np.mean(y[ids] / 100.)) if condition == 'MLP_TARGET_CENTER' else 0.
    return scaler, offset


def transform_features(X, scaler):
    values = np.asarray(X, dtype=np.float32)
    return values if scaler is None else scaler.transform(values).astype(np.float32)


def readout_metrics(y, prediction, cfg):
    result = metrics(np.asarray(y), np.asarray(prediction), cfg)
    errors = np.asarray(prediction) - np.asarray(y)
    result.update(MAE=float(np.abs(errors).mean()), bias=float(errors.mean()),
                  max_absolute_error=float(np.abs(errors).max()), n=len(errors),
                  rounded_exact=int(np.sum(np.ceil(np.clip(prediction, 0, 100) - .5) == y)))
    for threshold in (1, 2, 5, 10):
        result['within' + str(threshold)] = int(np.sum(np.abs(errors) <= threshold))
    return result


def metric_row(condition, seed, fold, y, p, cfg, **extra):
    m = readout_metrics(y, p, cfg)
    reasons = m.pop('undefined_reasons')
    return dict(condition=condition, seed=seed, fold=fold, **m,
                undefined_reasons=json.dumps(reasons, ensure_ascii=False), **extra)


def make_readout_head(seed, dropout):
    # Same initialization across conditions for the same seed/fold.
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    return nn.Sequential(nn.Linear(1024, 256), nn.GELU(), nn.Dropout(dropout), nn.Linear(256, 1))

In [ ]:
%%aes
def read_progress(path, contract):
    if not path.exists():
        return None
    index = json.loads(path.read_text())
    if index['contract'] != contract:
        raise ValueError('Checkpoint berasal dari data/fitur/konfigurasi/kode yang berbeda.')
    for item in index['snapshots']:
        if Path(item['file']).name != item['file']:
            raise ValueError('Nama checkpoint tidak sah.')
        file = path.parent / item['file']
        if file.is_file() and digest(file) == item['sha256']:
            return torch.load(file, map_location='cpu', weights_only=True)
    raise RuntimeError('Kedua checkpoint lokal tidak utuh. Jangan menimpa hasil lama.')


def write_progress(path, payload, contract):
    path.parent.mkdir(parents=True, exist_ok=True)
    index = json.loads(path.read_text()) if path.exists() else {'snapshots': []}
    slot = 1 - index.get('slot', 1)
    file = path.with_name(path.stem + '_slot' + str(slot) + '.pt')
    temp = file.with_suffix('.tmp'); torch.save(payload, temp); temp.replace(file)
    snapshots = [dict(file=file.name, sha256=digest(file))]
    snapshots += [v for v in index['snapshots'] if v['file'] != file.name][:1]
    dump(path, dict(contract=contract, slot=slot, snapshots=snapshots))


def fit_readout(condition, X, y, training, validation, cfg, diagnostic, seed, path, contract):
    """CPU float32 head, identical optimization budget and stopping rule."""
    if condition not in CONDITIONS[:3]:
        raise ValueError('fit_readout khusus kondisi MLP.')
    tr = np.asarray(training, dtype=int); va = np.asarray(validation, dtype=int)
    if not len(tr) or not len(va) or set(tr) & set(va):
        raise ValueError('Train/validation harus terpisah dan tidak kosong.')
    scaler, offset = prepare_transform(X, y, tr, condition)
    tx = torch.from_numpy(transform_features(X, scaler))
    targets = torch.as_tensor(y / 100. - offset, dtype=torch.float32).reshape(-1, 1)
    head = make_readout_head(seed, cfg['dropout'])
    initial_hash = hashlib.sha256(b''.join(t.detach().numpy().tobytes() for t in head.parameters())).hexdigest()
    optimizer = torch.optim.AdamW(head.parameters(), lr=diagnostic['lr'], weight_decay=cfg['weight_decay'])
    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(tr.tolist(), batch_size=cfg['effective_batch'], shuffle=True, generator=generator)
    total = len(loader) * cfg['epochs']
    warmup = min(int(cfg['warmup_ratio'] * total), max(0, total - 1))
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup, total)
    start, best, best_epoch, stale, history, best_state = 1, float('inf'), 0, 0, [], None
    saved = read_progress(path, contract)
    if saved:
        if saved['complete']:
            return saved
        head.load_state_dict(saved['state']); optimizer.load_state_dict(saved['optimizer'])
        scheduler.load_state_dict(saved['scheduler']); generator.set_state(saved['generator'])
        torch.set_rng_state(saved['torch_rng'])
        start, best, best_epoch, stale = saved['epoch'] + 1, saved['best'], saved['best_epoch'], saved['stale']
        history, best_state = saved['history'], saved['best_state']
    for epoch in range(start, cfg['epochs'] + 1):
        head.train(); loss_sum = 0.; steps = 0
        for effective in loader:
            ids = effective.tolist(); optimizer.zero_grad(set_to_none=True)
            for begin in range(0, len(ids), cfg['micro_batch']):
                batch = ids[begin:begin + cfg['micro_batch']]
                loss = (head(tx[batch]) - targets[batch]).square().mean()
                if not torch.isfinite(loss):
                    raise RuntimeError('Loss nonfinite; tidak dianggap hasil berhasil.')
                (loss * (len(batch) / len(ids))).backward()
                loss_sum += loss.item() * len(batch)
            torch.nn.utils.clip_grad_norm_(head.parameters(), cfg['gradient_clip'])
            optimizer.step(); scheduler.step(); steps += 1
        head.eval()
        with torch.no_grad():
            prediction = (head(tx[va]).reshape(-1).numpy() + offset) * 100.
        score = readout_metrics(y[va], prediction, cfg)['RMSE']
        if score < best:
            best, best_epoch, stale = score, epoch, 0
            best_state = {k: v.detach().cpu().clone() for k, v in head.state_dict().items()}
        else:
            stale += 1
        history.append(dict(epoch=epoch, train_mse_scaled=loss_sum / len(tr), validation_RMSE=score,
                            best_epoch=best_epoch, lr_end=optimizer.param_groups[0]['lr'],
                            optimizer_steps=steps, target_offset=offset))
        complete = stale >= cfg['early_stopping_patience'] or epoch == cfg['epochs']
        saved = dict(complete=complete, epoch=epoch, state=head.state_dict(), optimizer=optimizer.state_dict(),
                     scheduler=scheduler.state_dict(), generator=generator.get_state(), torch_rng=torch.get_rng_state(),
                     best=best, best_epoch=best_epoch, stale=stale, history=history, best_state=best_state,
                     initial_parameter_sha256=initial_hash, target_offset=offset,
                     scaler_mean=None if scaler is None else torch.from_numpy(scaler.mean_),
                     scaler_scale=None if scaler is None else torch.from_numpy(scaler.scale_))
        if complete:
            head.load_state_dict(best_state)
            with torch.no_grad():
                saved['prediction'] = torch.from_numpy(((head(tx[va]).reshape(-1).numpy() + offset) * 100.).copy())
            saved['state'] = best_state
        write_progress(path, saved, contract)
        print(f'{condition} | seed {seed} | epoch {epoch}/{cfg["epochs"]} | RMSE validasi {score:.4f} | terbaik {best:.4f}', flush=True)
        if complete:
            break
    return saved

In [ ]:
%%aes
def ridge_fold(X, y, training, validation, folds, alphas):
    """Alpha never selected on the outer validation or historical test."""
    tr = np.asarray(training, dtype=int); va = np.asarray(validation, dtype=int)
    if set(tr) & set(va):
        raise ValueError('Outer train/validation bertumpang tindih.')
    grid = []
    for alpha in alphas:
        truth, predictions = [], []
        for fold in folds:
            iv = np.intersect1d(tr, fold['validation'])
            it = np.setdiff1d(tr, iv)
            if not len(iv):
                continue
            if not len(it):
                raise ValueError('Inner CV tidak cukup blok.')
            model = Ridge(alpha=alpha, fit_intercept=True, solver='cholesky').fit(X[it], y[it])
            truth.extend(y[iv]); predictions.extend(model.predict(X[iv]))
        if len(truth) != len(tr):
            raise ValueError('Inner OOF tidak meliputi seluruh outer training tepat sekali.')
        score = float(np.sqrt(np.mean((np.asarray(truth) - np.asarray(predictions)) ** 2)))
        grid.append(dict(alpha=alpha, inner_RMSE=score, inner_rows=len(truth)))
    chosen = min(grid, key=lambda row: row['inner_RMSE'])['alpha']
    model = Ridge(alpha=chosen, fit_intercept=True, solver='cholesky').fit(X[tr], y[tr])
    return model.predict(X[va]), chosen, grid


def run_readout_cv(bundle, features, manifest, cfg, diagnostic, folder):
    X = np.asarray(features, dtype=np.float32); y = np.asarray(bundle.y, dtype=np.float64)
    if X.shape != (len(y), 1024) or not np.isfinite(X).all():
        raise ValueError('Vektor BGE harus [jumlah_jawaban,1024] dan finite.')
    if not np.allclose(np.linalg.norm(X, axis=1), 1., atol=1e-4):
        raise ValueError('Kontrol memerlukan BGE L2 asli.')
    dev, test = set(manifest['development']), set(manifest['test'])
    feature_hash = hashlib.sha256(X.tobytes()).hexdigest()
    semantic = dict(source=bundle.source_sha256, split=manifest['split_sha256'], features=feature_hash,
                    config=cfg, diagnostic=diagnostic, code=NOTEBOOK_CODE_SHA256,
                    torch=str(torch.__version__), precision='cpu_float32', sklearn=version('scikit-learn'))
    key = hashlib.sha256(json.dumps(semantic, sort_keys=True).encode()).hexdigest()[:20]
    run = Path(folder) / key; run.mkdir(parents=True, exist_ok=True); dump(run / 'contract.json', semantic)
    evaluation, histories, predictions, transforms, grids = [], [], [], [], []
    for condition in CONDITIONS:
        seeds = [None] if condition == 'RIDGE' else diagnostic['seeds']
        for seed in seeds:
            oof = np.full(len(y), np.nan)
            for fold in manifest['folds']:
                tr, va = fold['train'], fold['validation']; number = fold['fold']
                if set(tr) & test or set(va) & test or set(tr) | set(va) != dev:
                    raise ValueError('Test memasuki proses CV atau partisi tidak lengkap.')
                if set(bundle.groups[tr]) & set(bundle.groups[va]):
                    raise ValueError('Kelompok peserta menyeberang fold.')
                if condition == 'RIDGE':
                    p, alpha, search = ridge_fold(X, y, tr, va, manifest['folds'], diagnostic['alphas'])
                    grids.extend(dict(fold=number, **row) for row in search)
                    transforms.append(dict(condition=condition, seed=seed, fold=number, target_offset=0., chosen_alpha=alpha))
                    best_epoch = None
                else:
                    contract = dict(key=key, condition=condition, seed=seed, fold=number)
                    state = fit_readout(condition, X, y, tr, va, cfg, diagnostic, seed + number,
                                        run / condition / str(seed) / f'fold_{number}.json', contract)
                    p = state['prediction'].numpy(); best_epoch = state['best_epoch']
                    histories.extend(dict(condition=condition, seed=seed, fold=number, **row) for row in state['history'])
                    transforms.append(dict(condition=condition, seed=seed, fold=number, target_offset=state['target_offset'],
                                           initial_parameter_sha256=state['initial_parameter_sha256'], best_epoch=best_epoch))
                    if state['scaler_mean'] is not None:
                        transforms.extend(dict(condition=condition, seed=seed, fold=number, component=i + 1,
                                               mean=float(m), scale=float(s)) for i, (m, s) in enumerate(zip(state['scaler_mean'], state['scaler_scale'])))
                if np.isfinite(oof[va]).any():
                    raise ValueError('Jawaban validasi diprediksi lebih dari sekali.')
                oof[va] = p
                evaluation.append(metric_row(condition, seed, number, y[va], p, cfg, best_epoch=best_epoch))
                predictions.extend(dict(condition=condition, seed=seed, fold=number, row_id=bundle.rows[i]['row_id'],
                                        teacher=float(y[i]), prediction=float(v), error=float(v - y[i]),
                                        absolute_error=float(abs(v - y[i]))) for i, v in zip(va, p))
            dv = np.asarray(sorted(dev))
            if not np.isfinite(oof[dv]).all() or np.isfinite(oof[list(test)]).any():
                raise ValueError('OOF tidak lengkap atau mengandung prediksi test.')
            evaluation.append(metric_row(condition, seed, 'POOLED_OOF', y[dv], oof[dv], cfg))
    result = dict(run_folder=str(run), feature_sha256=feature_hash, evaluation=evaluation, history=histories,
                  predictions=predictions, transforms=transforms, ridge_grid=grids,
                  test_evaluated=False, BGE_encoder_trained=False, head_device='cpu', head_precision='float32')
    dump(run / 'readout_result.json', result)
    return result

In [ ]:
%%aes
def summarize_readouts(result, tolerance):
    scores = pd.DataFrame([r for r in result['evaluation'] if r['fold'] == 'POOLED_OOF'])
    rows = []
    for condition in CONDITIONS:
        group = scores[scores.condition == condition]; n = len(group)
        rows.append(dict(condition=condition, repetitions=n, n_per_repetition=int(group.n.iloc[0]),
                         RMSE_mean=float(group.RMSE.mean()), RMSE_seed_sd=float(group.RMSE.std(ddof=1)) if n > 1 else None,
                         MAE_mean=float(group.MAE.mean()), within1_mean=float(group.within1.mean()),
                         within2_mean=float(group.within2.mean()), within5_mean=float(group.within5.mean()),
                         primary_within_mean=float(group['within' + str(tolerance)].mean())))
    for row in rows:
        row['pareto_RMSE_and_tolerance'] = not any(
            other['RMSE_mean'] <= row['RMSE_mean'] and other['primary_within_mean'] >= row['primary_within_mean']
            and (other['RMSE_mean'] < row['RMSE_mean'] or other['primary_within_mean'] > row['primary_within_mean'])
            for other in rows)
    return pd.DataFrame(rows)


def readout_duels(result, diagnostic):
    rows = []; data = pd.DataFrame(result['predictions'])
    for seed in diagnostic['seeds']:
        selected = data[(data.seed == seed) | (data.condition == 'RIDGE')]
        for left, right in itertools.combinations(CONDITIONS, 2):
            a = selected[selected.condition == left].set_index('row_id')
            b = selected[selected.condition == right].set_index('row_id')
            if set(a.index) != set(b.index):
                raise ValueError('Duel tidak memakai jawaban yang sama.')
            diff = a.absolute_error - b.loc[a.index].absolute_error
            rows.append(dict(seed=seed, left=left, right=right, n=len(diff),
                             left_closer=int((diff < -1e-5).sum()), right_closer=int((diff > 1e-5).sum()),
                             ties=int((diff.abs() <= 1e-5).sum())))
    return pd.DataFrame(rows)


def write_readout_workbook(path, sheets):
    # Strings are literal text, never spreadsheet formulas from uploaded essays.
    book = Workbook(write_only=True)
    for title, data in sheets.items():
        frame = data if isinstance(data, pd.DataFrame) else pd.DataFrame(data)
        sheet = book.create_sheet(title[:31])
        sheet.append([str(v) for v in frame.columns])
        for record in frame.itertuples(index=False, name=None):
            cells = []
            for value in record:
                if isinstance(value, (list, dict, tuple)):
                    value = json.dumps(value, ensure_ascii=False)
                if isinstance(value, np.generic):
                    value = value.item()
                if isinstance(value, float) and not math.isfinite(value):
                    value = None
                cell = WriteOnlyCell(sheet, value=value)
                if isinstance(value, str):
                    cell.data_type = 's'
                cells.append(cell)
            sheet.append(cells)
    book.save(path)


def export_readout(bundle, features, manifest, cfg, diagnostic, result):
    run = Path(result['run_folder']); destination = run / 'hasil_ringkas'; destination.mkdir(exist_ok=True)
    roles = {i: 'development' for i in manifest['development']}; roles.update({i: 'test_locked' for i in manifest['test']})
    records = pd.DataFrame([dict(row_id=r['row_id'], soal=r['question'], jawaban_referensi=r['reference'],
                                 response_text=r['text'], rater=r['score'], connected_group=int(bundle.groups[i]),
                                 role=roles[i]) for i, r in enumerate(bundle.rows)])
    split = pd.DataFrame([dict(fold=f['fold'], role=role, row_id=bundle.rows[i]['row_id'])
                          for f in manifest['folds'] for role in ('train', 'validation') for i in f[role]])
    guide = pd.DataFrame([dict(Keterangan=k, Nilai=v) for k, v in dict(
        scope='DIAGNOSTIK BGE FROZEN; empat kondisi readout; bukan S1/S2/S3 lengkap',
        target='rater 0–100; tiga input; data guru asli',
        test='Tidak dievaluasi; seleksi hanya development/OOF',
        seed='Ridge deterministik dihitung sekali; mean seed bukan ensemble prediksi',
        comparison='Target-centering dan StandardScaler terpisah; statistik train fold saja',
        precision='Hitungan toleransi dan RMSE dilaporkan terpisah; Pareto bukan klaim satu pemenang',
        training='MSE target0–1; CPU float32; BGE tidak mendapat optimizer',
        privacy='Seluruh Excel/cache/checkpoint privat; jangan unggah ke repo publik').items()])
    feature_frame = pd.DataFrame(np.asarray(features), columns=[f'Komponen {i+1}' for i in range(1024)])
    feature_frame.insert(0, 'row_id', records.row_id)
    summary = summarize_readouts(result, diagnostic['primary_tolerance'])
    write_readout_workbook(destination / REPORT_BOOKS[0], dict(Panduan=guide, Data=records, Lima_Fold=split))
    write_readout_workbook(destination / REPORT_BOOKS[1], dict(Panduan=guide, BGE_Frozen=feature_frame,
                                                            Transformasi_Train_Fold=pd.DataFrame(result['transforms'])))
    write_readout_workbook(destination / REPORT_BOOKS[2], dict(Panduan=guide, Prediksi_Development_OOF=pd.DataFrame(result['predictions'])))
    write_readout_workbook(destination / REPORT_BOOKS[3], dict(Panduan=guide, Ringkasan=summary,
        Fold_dan_Pooled=pd.DataFrame(result['evaluation']), Duel=readout_duels(result, diagnostic), Ridge_Inner_CV=pd.DataFrame(result['ridge_grid'])))
    parameters = dict(**cfg, **diagnostic, code_sha256=NOTEBOOK_CODE_SHA256, feature_sha256=result['feature_sha256'])
    write_readout_workbook(destination / REPORT_BOOKS[4], dict(Panduan=guide,
        Pengaturan=pd.DataFrame([dict(parameter=k, value=v) for k,v in parameters.items()]),
        Riwayat_MLP=pd.DataFrame(result['history']),
        Versi=pd.DataFrame([dict(library=k, version=version(k)) for k in ('torch','transformers','numpy','pandas','scikit-learn','openpyxl')])))
    archive = run / 'Hasil_AES_300_BGE_Frozen_Readout.zip'
    with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
        for name in REPORT_BOOKS:
            z.write(destination / name, name)
        z.writestr('manifest.json', json.dumps(dict(scope=guide.iloc[0].Nilai, dataset_sha256=bundle.source_sha256,
            code_sha256=NOTEBOOK_CODE_SHA256, feature_sha256=result['feature_sha256'], test_evaluated=False,
            books={n:digest(destination / n) for n in REPORT_BOOKS}), ensure_ascii=False, indent=2))
    return dict(archive=str(archive), books=list(REPORT_BOOKS), run_folder=str(run), test_evaluated=False)

In [ ]:
%%aes
diagnostic = readout_config(diagnostic)
show(pd.DataFrame([dict(kondisi=k, penjelasan=DESCRIPTIONS[k]) for k in CONDITIONS]), 'Perlakuan yang terpisah')
print('Seed MLP:',diagnostic['seeds'],'| LR:',diagnostic['lr'],'| Ridge alpha:',diagnostic['alphas'],flush=True)

## 8 · Jalankan validasi berpasangan

Semua kondisi memakai jawaban dan fold yang sama. MLP1024→256→1 memakai AdamW, MSE target0–1, warmup10%, clipping1 dan patience5. Checkpoint dua slot menyimpan progres epoch dan RNG; saat folder masih ada, proses dapat dilanjutkan. Perubahan kode/data/config membuat folder hasil berbeda.

Untuk setiap outer fold Ridge, alpha dipilih tanpa outer validation. Tidak ada pencarian parameter dari test. Riwayat training MSE mencakup dropout dan perubahan bobot; jangan dianggap sama dengan train RMSE saat evaluasi.

In [ ]:
%%aes
result = run_readout_cv(bundle, X, manifest, cfg, diagnostic, output/'readout_runs')
print('Seluruh CV selesai. Test dievaluasi:',result['test_evaluated'],flush=True)

## 9 · Baca hasil dan perpindahan prediksi

Tabel pooled OOF mencakup seluruh development: setiap jawaban dibaca oleh head yang tidak dilatih pada jawaban/kelompoknya. Ringkasan antar-seed bukan penggabungan prediksi menjadi ensemble. Pareto menandai kandidat yang tidak dikalahkan sekaligus pada RMSE dan jumlah dekat; bukan memilih satu pemenang berdasarkan test.

In [ ]:
%%aes
summary = summarize_readouts(result, diagnostic['primary_tolerance'])
show(summary, 'Ringkasan empat kondisi; jumlah dekat per development, bukan persentase')
show(pd.DataFrame([r for r in result['evaluation'] if r['fold']=='POOLED_OOF']), 'OOF setiap seed')
show(readout_duels(result, diagnostic), 'Duel pada jawaban development yang sama')
show(pd.DataFrame(result['history']), 'Riwayat MLP; rincian lengkap di Excel')
print('Jika error turun tetapi jumlah dekat turun juga, catat sebagai tradeoff. Jangan menyimpulkan perbaikan S3.',flush=True)

## 10 · Lima Excel asli dan ZIP unduhan

Laporan mencakup dataset/split, fitur BGE/transformasi train fold, prediksi OOF, evaluasi/duel, serta riwayat. File berisi data privat. ZIP berisi lima Excel dan manifest checksum; bobot/cache/checkpoint tetap lokal dan tidak dimasukkan ke repo publik.

**Tidak ada hasil test atau hasil S1/S3 yang dibuat.** Ini menyelesaikan rencana diagnosis pertama. Refit/test dan integrasi S3 merupakan pengujian berikutnya setelah resep dipilih dari development.

In [ ]:
%%aes
download_info = export_readout(bundle, X, manifest, cfg, diagnostic, result)
dump(ROOT/'outputs/bge_readout_download.json', download_info)
show(pd.DataFrame([download_info]), 'Paket laporan privat siap')

In [ ]:
info=json.loads((ROOT/'outputs/bge_readout_download.json').read_text())
archive=Path(info['archive'])
if not archive.is_file():raise FileNotFoundError('Paket belum selesai; periksa cell sebelumnya.')
print('ZIP hasil:',archive)
print('Test tidak dievaluasi. Tidak ada klaim checkpoint pretrained sudah dilatih.')
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))
else:
    print('Jalankan di Colab untuk unduhan browser, atau ambil file lokal di path di atas.')